# Model 2 - ResNet
Notebook train và test ResNet từ đầu.

In [ ]:
# 1. Import và cấu hình
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / 'data').exists() and PROJECT_DIR != PROJECT_DIR.parent:
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / 'data' / 'VN_trash_classification_preprocessing'
CHECKPOINT_DIR = PROJECT_DIR / 'src' / 'model2' / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2. Xử lý ảnh và load dữ liệu
norm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    norm,
])
test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    norm,
])

train_ds = ImageFolder(DATA_DIR / 'train', transform=train_tf)
val_ds = ImageFolder(DATA_DIR / 'val', transform=test_tf)
test_ds = ImageFolder(DATA_DIR / 'test', transform=test_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

class_names = train_ds.classes
print('Device:', device, '| Classes:', class_names)
print(len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# 3. Tạo ResNet
class BasicBlock(nn.Module):
    def __init__(self, in_channels, channels, stride=1):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, channels, 3, stride, 1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        if stride != 1 or in_channels != channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, channels, 1, stride, bias=False),
                nn.BatchNorm2d(channels),
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        shortcut = self.shortcut(x)
        output = self.conv2(self.conv1(x))
        return self.relu(output + shortcut)


class ResNet(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # 4 stage, mỗi stage 2 BasicBlock
        self.layer1 = nn.Sequential(BasicBlock(64, 64, 1), BasicBlock(64, 64))
        self.layer2 = nn.Sequential(BasicBlock(64, 128, 2), BasicBlock(128, 128))
        self.layer3 = nn.Sequential(BasicBlock(128, 256, 2), BasicBlock(256, 256))
        self.layer4 = nn.Sequential(BasicBlock(256, 512, 2), BasicBlock(512, 512))

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, number_of_classes),
        )

    def forward(self, images):
        x = self.stem(images)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.classifier(self.pool(x))


model = ResNet(len(class_names)).to(device)
print(model)

In [ ]:
# 4. Loss, optimizer và scheduler
counts = Counter(train_ds.targets)
weights = torch.tensor(
    [len(train_ds) / (len(class_names) * counts[i]) for i in range(len(class_names))],
    dtype=torch.float32,
    device=device,
)

loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

In [ ]:
# 5. Train và validation
def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        seen += labels.size(0)

    return total_loss / seen, correct / seen


best_acc = 0
best_state = None

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader)
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch + 1}/{EPOCHS} | Train: {train_acc:.2%} | Val: {val_acc:.2%}')

model.load_state_dict(best_state)
print('Best validation accuracy:', f'{best_acc:.2%}')

In [ ]:
# 6. Test và đánh giá
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        true_labels += labels.tolist()
        predicted_labels += outputs.argmax(1).cpu().tolist()

accuracy = sum(t == p for t, p in zip(true_labels, predicted_labels)) / len(true_labels)
mp, mr, mf, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='macro', zero_division=0)
_, _, wf, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='weighted', zero_division=0)

print('Test Accuracy:', f'{accuracy:.2%}')
print('Macro Precision:', f'{mp:.4f}')
print('Macro Recall:', f'{mr:.4f}')
print('Macro F1:', f'{mf:.4f}')
print('Weighted F1:', f'{wf:.4f}')
print(confusion_matrix(true_labels, predicted_labels))

In [ ]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR / 'resnet.pt')